# Milestone 4.35: Identifying and Removing Duplicate Records

**Objective:** Master the essential skill of detecting and removing duplicate records to ensure your dataset represents unique, reliable observations.

**Key Learning:**
- Duplicate data is a common quality issue that skews analysis
- **Detect** = Find repeated rows using `duplicated()`
- **Remove** = Drop duplicates using `drop_duplicates()`
- **Verify** = Always check results after deduplication
- **Think carefully** = Not all duplicates should be removed blindly

**Think of it this way:** Duplicate records are like counting the same person twice in a survey—it inflates numbers and distorts reality.

---

## Why Duplicate Records Matter

### The Problem

**Common beginner mistakes:**
- ❌ Counting duplicate records as unique observations → Inflated metrics
- ❌ Removing duplicates blindly without inspection
- ❌ Not understanding why duplicates exist
- ❌ Losing important information during deduplication
- ❌ Not verifying results after removing duplicates

**Result:** Analysis based on distorted data → Wrong counts, misleading summaries, bad decisions.

### The Solution

**Three-step process:**

1. **Detect Duplicates**
   - Use `duplicated()` to find repeated rows
   - Inspect which records are duplicated
   - Understand the extent of duplication

2. **Remove Duplicates**
   - Use `drop_duplicates()` to remove repeated rows
   - Choose which duplicate to keep (first, last, or none)
   - Apply to specific columns if needed

3. **Verify Results**
   - Check shape before and after
   - Confirm no duplicates remain
   - Ensure important records preserved

**Professional habit:** Always inspect duplicates before removing them—understand the context!

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

---

## Part 1: Understanding Duplicate Records

### What Are Duplicate Records?

**Exact duplicates:** Rows where ALL values are identical

In [ ]:
# Example: Dataset with exact duplicates
data = {
    'Name': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'Age': [25, 30, 25, 35, 30],
    'City': ['NYC', 'LA', 'NYC', 'Chicago', 'LA']
}
df_example = pd.DataFrame(data)

print("Dataset with duplicates:")
print(df_example)
print(f"\nShape: {df_example.shape}")

**Observation:**
- Row 0 and Row 2: Same Name, Age, City → **Exact duplicate**
- Row 1 and Row 4: Same Name, Age, City → **Exact duplicate**
- Row 3: Unique

**Result:** 5 rows, but only 3 unique records!

### Why Do Duplicates Occur?

**Common causes:**

1. **Data entry errors** - Same record entered multiple times
2. **System errors** - Repeated API calls, duplicate form submissions
3. **Merge operations** - Joining datasets creates duplicates
4. **Multiple sources** - Same data collected from different systems
5. **Time-based records** - Person appears multiple times over time

**Important:** Not all duplicates are mistakes!
- Customer making multiple purchases → Keep all (different transactions)
- Same customer record appearing twice → Remove duplicate

### Loading Real Data with Duplicates

In [ ]:
# Load customer data
customers_df = pd.read_csv('../data/raw/customer_data.csv')

print("Customer Data:")
print(customers_df)
print(f"\nShape: {customers_df.shape}")
print(f"Total rows: {len(customers_df)}")

**Question:** Are there duplicates? Let's find out!

---

## Part 2: Detecting Duplicate Rows

### Method 1: Using duplicated() to Mark Duplicates

**`duplicated()`** returns a boolean Series where `True` indicates a duplicate row.

In [ ]:
# Check for duplicate rows
is_duplicate = customers_df.duplicated()

print("Duplicate indicators:")
print(is_duplicate)
print(f"\nNumber of duplicate rows: {is_duplicate.sum()}")
print(f"Percentage of duplicates: {(is_duplicate.sum() / len(customers_df) * 100):.1f}%")

**Key insight:** 
- `False` = First occurrence (original record)
- `True` = Subsequent occurrences (duplicates)

**By default, `duplicated()` marks all duplicates EXCEPT the first occurrence.**

### Method 2: Viewing All Duplicate Rows (Including First Occurrence)

In [ ]:
# Show ONLY the rows that are duplicates (marked as True)
print("Duplicate rows (excluding first occurrence):")
print(customers_df[is_duplicate])

**Problem:** This doesn't show the original (first occurrence)!

**Better approach:** Show ALL rows involved in duplication (including originals)

In [ ]:
# Get rows that have duplicates (including first occurrence)
# Use keep=False to mark ALL duplicates (including first)
all_duplicates = customers_df[customers_df.duplicated(keep=False)]

print("All rows involved in duplication (including originals):")
print(all_duplicates.sort_values('CustomerID'))

**Much better!** Now we see:
- Which customers appear multiple times
- How many times each appears
- Full context of duplication

**✅ Always use `keep=False` when inspecting duplicates for full picture.**

### Method 3: Counting Duplicate Occurrences

In [ ]:
# Count how many times each unique row appears
print("Duplicate counts by CustomerID:")
duplicate_counts = customers_df['CustomerID'].value_counts()
print(duplicate_counts)

print("\nCustomers with duplicates (appear > 1 time):")
print(duplicate_counts[duplicate_counts > 1])

**Key insight:** This shows which specific values are duplicated and how many times.

### Method 4: Detecting Duplicates Based on Specific Columns

**Sometimes you only care about duplicates in certain columns, not all columns.**

In [ ]:
# Check duplicates based only on CustomerID and Email
# (ignore other columns like SignupDate)
duplicates_by_id = customers_df.duplicated(subset=['CustomerID', 'Email'])

print(f"Duplicates based on CustomerID + Email: {duplicates_by_id.sum()}")

# Show these duplicates
print("\nRows with duplicate CustomerID and Email:")
print(customers_df[customers_df.duplicated(subset=['CustomerID', 'Email'], keep=False)].sort_values('CustomerID'))

**Use case:** When you want to identify duplicates based on key identifiers only.

**Examples:**
- Duplicate customers: Check `CustomerID` or `Email`
- Duplicate transactions: Check `TransactionID`
- Duplicate products: Check `SKU` or `ProductCode`

---

## Part 3: Removing Duplicate Records

### Method 1: Remove All Duplicates (Keep First Occurrence)

**`drop_duplicates()`** removes duplicate rows.

In [ ]:
# Remove duplicates (keep first occurrence by default)
customers_dedup = customers_df.drop_duplicates()

print("After removing duplicates:")
print(customers_dedup)
print(f"\nOriginal shape: {customers_df.shape}")
print(f"After deduplication: {customers_dedup.shape}")
print(f"Rows removed: {len(customers_df) - len(customers_dedup)}")

**Result:** Kept only unique rows!

**Behavior:**
- First occurrence of each duplicate group is **kept**
- Subsequent duplicates are **removed**
- Original DataFrame is **not modified** (returns new DataFrame)

### Method 2: Remove Duplicates (Keep Last Occurrence)

In [ ]:
# Keep last occurrence instead of first
customers_keep_last = customers_df.drop_duplicates(keep='last')

print("Keep last occurrence:")
print(customers_keep_last)
print(f"\nShape: {customers_keep_last.shape}")

**When to use `keep='last'`:**
- When the most recent record is more up-to-date
- When duplicates represent updates to existing records
- When you want the latest version of data

**Example:** Customer updates their email—keep the last one.

### Method 3: Remove All Occurrences of Duplicates

In [ ]:
# Remove ALL duplicates (don't keep any)
customers_remove_all = customers_df.drop_duplicates(keep=False)

print("Remove ALL duplicate occurrences:")
print(customers_remove_all)
print(f"\nShape: {customers_remove_all.shape}")
print(f"Rows removed: {len(customers_df) - len(customers_remove_all)}")

**When to use `keep=False`:**
- When you only want records that are unique
- When you can't determine which duplicate is correct
- When duplicates indicate problematic data

**⚠️ Warning:** This is aggressive—use carefully!

### Method 4: Remove Duplicates Based on Specific Columns

**Most common real-world approach:** Identify duplicates based on key columns only.

In [ ]:
# Remove duplicates based on CustomerID only
# (Keep first occurrence of each CustomerID)
customers_unique_id = customers_df.drop_duplicates(subset=['CustomerID'])

print("One row per CustomerID:")
print(customers_unique_id)
print(f"\nOriginal rows: {len(customers_df)}")
print(f"Unique customers: {len(customers_unique_id)}")
print(f"Duplicate customer records removed: {len(customers_df) - len(customers_unique_id)}")

**✅ This is often the best approach!**

**Why?**
- Focus on business keys (ID, Email, etc.)
- Ignore minor differences in other columns
- Preserve one record per unique entity

**Examples:**
- `subset=['CustomerID']` → One row per customer
- `subset=['Email']` → One row per email address
- `subset=['TransactionID']` → One row per transaction

### Method 5: Remove Duplicates and Modify Original DataFrame

In [ ]:
# Create a copy to demonstrate inplace modification
customers_copy = customers_df.copy()

print(f"Before: {customers_copy.shape}")

# Remove duplicates and modify DataFrame in place
customers_copy.drop_duplicates(inplace=True)

print(f"After: {customers_copy.shape}")
print("\n✅ Original DataFrame modified!")

**⚠️ Caution with `inplace=True`:**
- Modifies the original DataFrame
- Cannot undo the operation
- Prefer creating new DataFrame for safety

**Recommendation:** Avoid `inplace=True` unless you're sure.

---

## Part 4: Comparing Keep Strategies

### Side-by-Side Comparison

In [ ]:
# Create a simple example to see the difference
test_data = pd.DataFrame({
    'ID': [1, 1, 2, 3, 3, 3],
    'Value': ['A', 'B', 'C', 'D', 'E', 'F'],
    'Date': ['2024-01-01', '2024-01-02', '2024-01-03', 
             '2024-01-04', '2024-01-05', '2024-01-06']
})

print("Original data:")
print(test_data)
print(f"Shape: {test_data.shape}")

In [ ]:
# Strategy 1: Keep first
keep_first = test_data.drop_duplicates(subset=['ID'], keep='first')
print("Keep FIRST occurrence:")
print(keep_first)
print(f"Shape: {keep_first.shape}\n")

In [ ]:
# Strategy 2: Keep last
keep_last = test_data.drop_duplicates(subset=['ID'], keep='last')
print("Keep LAST occurrence:")
print(keep_last)
print(f"Shape: {keep_last.shape}\n")

In [ ]:
# Strategy 3: Keep none
keep_none = test_data.drop_duplicates(subset=['ID'], keep=False)
print("Keep NO duplicates (only truly unique):")
print(keep_none)
print(f"Shape: {keep_none.shape}\n")

**Observations:**

- **`keep='first'`:** ID 1 keeps 'A', ID 3 keeps 'D' (earliest dates)
- **`keep='last'`:** ID 1 keeps 'B', ID 3 keeps 'F' (latest dates)
- **`keep=False`:** Removes ID 1 and ID 3 entirely (only ID 2 is unique)

**Decision guide:**
- Use `'first'` when order doesn't matter or you want earliest
- Use `'last'` when you want most recent update
- Use `False` when you only want truly unique records

---

## Part 5: Real-World Example - Transaction Data

### Loading Transaction Data

In [ ]:
# Load transaction data
transactions_df = pd.read_csv('../data/raw/transactions.csv')

print("Transaction Data:")
print(transactions_df)
print(f"\nShape: {transactions_df.shape}")

### Step 1: Detect Duplicates in Transactions

In [ ]:
# Check for duplicate rows
print(f"Total duplicate rows: {transactions_df.duplicated().sum()}")

# Show all duplicates
print("\nAll duplicate transactions:")
print(transactions_df[transactions_df.duplicated(keep=False)].sort_values('TransactionID'))

### Step 2: Analyze Duplicate Transactions

In [ ]:
# Count duplicates by TransactionID
print("Transaction IDs with duplicates:")
txn_counts = transactions_df['TransactionID'].value_counts()
print(txn_counts[txn_counts > 1].sort_index())

print(f"\nUnique transactions: {transactions_df['TransactionID'].nunique()}")
print(f"Total rows: {len(transactions_df)}")
print(f"Duplicate rows: {len(transactions_df) - transactions_df['TransactionID'].nunique()}")

**Finding:** Multiple rows with same TransactionID—likely data entry error or system glitch.

### Step 3: Remove Duplicate Transactions

In [ ]:
# Remove duplicates based on TransactionID (keep first)
transactions_clean = transactions_df.drop_duplicates(subset=['TransactionID'], keep='first')

print("Cleaned transactions:")
print(transactions_clean)
print(f"\nOriginal: {transactions_df.shape}")
print(f"Cleaned: {transactions_clean.shape}")
print(f"Duplicates removed: {len(transactions_df) - len(transactions_clean)}")

### Step 4: Verify Deduplication

In [ ]:
# Verify no duplicates remain
print("Verification:")
print(f"Remaining duplicates: {transactions_clean.duplicated(subset=['TransactionID']).sum()}")
print(f"Unique TransactionIDs: {transactions_clean['TransactionID'].nunique()}")
print(f"Total rows: {len(transactions_clean)}")

if transactions_clean['TransactionID'].nunique() == len(transactions_clean):
    print("\n✅ SUCCESS: Each TransactionID is unique!")
else:
    print("\n⚠️ WARNING: Duplicates still exist!")

### Step 5: Calculate Impact on Revenue

In [ ]:
# See how duplicates affected revenue calculation
revenue_with_dupes = transactions_df['Amount'].sum()
revenue_clean = transactions_clean['Amount'].sum()

print("Revenue Impact:")
print(f"Revenue with duplicates: ${revenue_with_dupes:,.2f}")
print(f"Revenue after deduplication: ${revenue_clean:,.2f}")
print(f"Overcount due to duplicates: ${revenue_with_dupes - revenue_clean:,.2f}")
print(f"Percentage overcount: {((revenue_with_dupes - revenue_clean) / revenue_clean * 100):.1f}%")

**Critical insight:** Duplicates inflated revenue by counting same transactions multiple times!

**This is why deduplication matters!**

---

## Part 6: Common Mistakes and Best Practices

### Mistake 1: Not Inspecting Before Removing

In [ ]:
# ❌ BAD: Remove duplicates blindly
# df_clean = df.drop_duplicates()  # What did we remove?

# ✅ GOOD: Inspect first
print("Before removing:")
print(f"Total rows: {len(customers_df)}")
print(f"Duplicates: {customers_df.duplicated().sum()}")
print(f"Unique customers: {customers_df['CustomerID'].nunique()}")

# Show what will be removed
print("\nRows to be removed:")
print(customers_df[customers_df.duplicated()])

# Then remove
customers_clean = customers_df.drop_duplicates()
print(f"\n✅ After: {len(customers_clean)} rows")

### Mistake 2: Using Wrong Subset of Columns

In [ ]:
# Consider this scenario
test_df = pd.DataFrame({
    'CustomerID': [1, 1, 2],
    'Email': ['john@email.com', 'john.doe@email.com', 'jane@email.com'],
    'Phone': ['555-1234', '555-1234', '555-5678']
})

print("Data:")
print(test_df)

# Wrong: Check all columns
wrong = test_df.drop_duplicates()  # No duplicates found (emails differ)
print(f"\n❌ Checking all columns: {len(wrong)} rows (missed duplicate customer!)")

# Right: Check CustomerID only
right = test_df.drop_duplicates(subset=['CustomerID'])
print(f"✅ Checking CustomerID: {len(right)} rows (correct!)")

**Lesson:** Choose the right columns that define uniqueness!

### Mistake 3: Not Verifying After Deduplication

In [ ]:
# Always verify your deduplication worked
def verify_deduplication(df, key_column):
    """Verify no duplicates remain in key column"""
    n_rows = len(df)
    n_unique = df[key_column].nunique()
    
    print(f"Total rows: {n_rows}")
    print(f"Unique {key_column}: {n_unique}")
    
    if n_rows == n_unique:
        print(f"✅ SUCCESS: All {key_column} are unique!")
        return True
    else:
        print(f"⚠️ WARNING: {n_rows - n_unique} duplicates still exist!")
        return False

# Use it
print("Verifying transactions:")
verify_deduplication(transactions_clean, 'TransactionID')

### Best Practice Checklist

In [ ]:
# Complete deduplication workflow
def deduplicate_workflow(df, key_columns, keep='first'):
    """
    Professional deduplication workflow with verification
    
    Parameters:
    -----------
    df : DataFrame
        Input DataFrame
    key_columns : list
        Columns to check for duplicates
    keep : str
        Which duplicate to keep ('first', 'last', or False)
    """
    print("=" * 60)
    print("DEDUPLICATION WORKFLOW")
    print("=" * 60)
    
    # Step 1: Inspect original data
    print(f"\n1. Original data: {df.shape}")
    
    # Step 2: Detect duplicates
    n_dupes = df.duplicated(subset=key_columns).sum()
    print(f"\n2. Duplicates detected: {n_dupes}")
    
    if n_dupes == 0:
        print("   ✅ No duplicates found!")
        return df
    
    # Step 3: Show duplicates
    print(f"\n3. Showing duplicate records:")
    dupes = df[df.duplicated(subset=key_columns, keep=False)]
    print(dupes[key_columns].head(10))
    
    # Step 4: Remove duplicates
    df_clean = df.drop_duplicates(subset=key_columns, keep=keep)
    print(f"\n4. After deduplication: {df_clean.shape}")
    print(f"   Rows removed: {len(df) - len(df_clean)}")
    
    # Step 5: Verify
    remaining = df_clean.duplicated(subset=key_columns).sum()
    print(f"\n5. Verification: {remaining} duplicates remaining")
    
    if remaining == 0:
        print("   ✅ Deduplication successful!")
    else:
        print("   ⚠️ WARNING: Duplicates still exist!")
    
    print("=" * 60)
    return df_clean

# Example usage
customers_final = deduplicate_workflow(customers_df, ['CustomerID'], keep='first')

---

## Summary: Key Takeaways

### Detection Methods

| Method | Purpose | Code |
|--------|---------|------|
| **Mark duplicates** | Find repeated rows | `df.duplicated()` |
| **Show all duplicates** | View originals + copies | `df[df.duplicated(keep=False)]` |
| **Count duplicates** | How many times each appears | `df['col'].value_counts()` |
| **Check specific columns** | Duplicates in key fields | `df.duplicated(subset=['ID'])` |

### Removal Methods

| Method | Behavior | Use Case |
|--------|----------|----------|
| **`keep='first'`** | Keep earliest occurrence | Default, most common |
| **`keep='last'`** | Keep latest occurrence | When latest is most current |
| **`keep=False`** | Remove all duplicates | When only truly unique wanted |
| **`subset=[...]`** | Check specific columns | Business key deduplication |

### Professional Workflow

```python
# 1. Inspect original data
print(f"Original: {df.shape}")

# 2. Detect duplicates
n_dupes = df.duplicated(subset=['ID']).sum()
print(f"Duplicates: {n_dupes}")

# 3. Show duplicates (optional but recommended)
print(df[df.duplicated(subset=['ID'], keep=False)])

# 4. Remove duplicates
df_clean = df.drop_duplicates(subset=['ID'], keep='first')

# 5. Verify
print(f"After: {df_clean.shape}")
print(f"Remaining duplicates: {df_clean.duplicated(subset=['ID']).sum()}")
```

### Critical Points

✅ **Always inspect duplicates before removing**

✅ **Use `keep=False` when inspecting** (shows all occurrences)

✅ **Use `subset=[...]` for business key deduplication**

✅ **Verify after deduplication** (check remaining duplicates)

✅ **Understand why duplicates exist** (error vs legitimate)

✅ **Document your deduplication decisions**

✅ **Check impact on metrics** (counts, sums affected)

### Common Mistakes to Avoid

❌ Removing duplicates blindly without inspection

❌ Using all columns when only key columns matter

❌ Not verifying after deduplication

❌ Using `inplace=True` without backup

❌ Assuming first occurrence is always correct

❌ Not understanding impact on analysis

### When NOT to Remove Duplicates

**Careful!** Not all repeated values are errors:

- **Customer purchases:** Same person buying multiple times → Keep all
- **Sensor readings:** Same value recorded multiple times → Keep all
- **Time series:** Repeated measurements → Keep all
- **Survey responses:** Multiple submissions allowed → Keep all

**Question to ask:** "Should this entity appear only once?"

---

## Next Steps

1. **Practice on different datasets**
2. **Always follow the 5-step workflow**
3. **Use `subset` parameter for business keys**
4. **Verify results after deduplication**
5. **Record your 2-minute demonstration video**

**You're now ready to handle duplicate records professionally!** 🎉